In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

# Lista Computacional 3

Esta lista cobre os capítulos [12 — Árvores de Decisão](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) e [13 — Bagging, Florestas e Boosting](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html).

## Como entregar

1. Abra este notebook no Colab pelo link do repositório.
2. `Arquivo → Salvar uma cópia no Drive`, antes de preencher qualquer coisa — é na sua cópia que você trabalha, não no original.
3. Preencha as células de resposta. As de texto são células de *markdown*: no Colab elas aparecem já formatadas, sem cursor — **dê um duplo clique** sobre a célula para abrir a edição, escreva por cima do texto que está lá e clique fora para formatar de novo.
4. **`Ambiente de execução → Executar tudo`**, de cima para baixo.
5. `Arquivo → Imprimir → Salvar como PDF`.
6. Poste o PDF no AVA/Moodle.

Sem o passo 4, o PDF sai sem nenhuma saída: gráfico nenhum, número nenhum, só as células que você escreveu.

E o `Executar tudo` **para na primeira célula que levantar erro**: dali para baixo nada é executado, e nenhuma dessas saídas entra no PDF. Se o PDF vier sem as saídas de um trecho, procure a célula que falhou, conserte e rode tudo de novo.

Antes de postar, confira também se alguma saída larga — uma tabela comprida, um gráfico grande — não ficou cortada na impressão.

## Prazo e peso

Entrega até **18/11/2026**. Peso: **10%** da nota.

## Identificação

Nome completo e matrícula:

*sua resposta aqui*

## Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix, mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold, train_test_split
from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    export_text,
    plot_tree,
)

## Exercício 1

*Do exercício 2 do capítulo 8 de James et al. (2023).*

A [seção 13.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/04-boosting.html) afirma que o boosting com árvores de uma divisão só, os tocos, leva a um modelo ***aditivo***, isto é, a um modelo da forma

$$
f(X) = \sum_{j=1}^{p} f_j(X_j).
$$

Explique por que isso acontece. Você pode começar pelo modelo final do algoritmo da seção 13.4, $\hat f(x) = \sum_{b=1}^{B} \lambda \hat f^b(x)$.

*sua resposta aqui*

## Exercício 2

*Do exercício 3 do capítulo 8 de James et al. (2023).*

Considere o índice de Gini, o erro de classificação e a entropia, da [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-arvores-de-classificacao.html), numa classificação com duas classes. Faça um único gráfico que mostre cada uma dessas três medidas em função de $\hat p_{m1}$. O eixo horizontal deve mostrar $\hat p_{m1}$, de 0 a 1, e o eixo vertical, o valor do índice de Gini, do erro de classificação e da entropia.

Dica: com duas classes, $\hat p_{m1} = 1 - \hat p_{m2}$.

In [ ]:
# sua resposta

## Exercício 3

*Do exercício 4 do capítulo 8 de James et al. (2023).*

Este exercício usa as duas figuras que a célula abaixo desenha. À esquerda, uma partição do espaço dos preditores $X_1$ e $X_2$ em regiões, como as da [seção 12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-arvores-de-regressao.html); os números dentro dos retângulos são a média de $Y$ em cada região. À direita, uma árvore; em cada nó, o ramo da esquerda é o das observações que satisfazem a condição, e o número em cada folha é a média de $Y$ nela. Execute a célula para ver as duas.

In [ ]:
fig, (painel_esq, painel_dir) = plt.subplots(1, 2, figsize=(10, 4))

# Painel da esquerda: a partição, com uma linha por corte e a média de cada região
painel_esq.plot([1, 1], [-1, 2.5], color="black")
painel_esq.plot([-1, 1], [1, 1], color="black")
painel_esq.plot([0, 0], [-1, 1], color="black")
painel_esq.plot([0, 1], [0, 0], color="black")
medias = [(0, 1.75, "15"), (2, 0.75, "5"), (-0.5, 0, "3"), (0.5, 0.5, "0"), (0.5, -0.5, "10")]
for x, y, media in medias:
    painel_esq.text(x, y, media, ha="center", va="center", fontsize=13)
painel_esq.set_xlim(-1, 3)
painel_esq.set_ylim(-1, 2.5)
painel_esq.set_xticks([0, 1])
painel_esq.set_yticks([0, 1])
painel_esq.set_xlabel("$X_1$")
painel_esq.set_ylabel("$X_2$")

# Painel da direita: a árvore, com a posição de cada nó e as ligações entre eles
nos = {
    "raiz": (0.5, 1.0, "$X_2 < 1$"),
    "a": (0.25, 0.6, "$X_1 < 1$"),
    "b": (0.75, 0.6, "$X_2 < 2$"),
    "c": (0.625, 0.2, "$X_1 < 0$"),
    "f1": (0.125, 0.2, "−1,80"),
    "f2": (0.375, 0.2, "0,63"),
    "f3": (0.53, -0.2, "−1,06"),
    "f4": (0.72, -0.2, "0,21"),
    "f5": (0.875, 0.2, "2,49"),
}
ligacoes = [("raiz", "a"), ("raiz", "b"), ("a", "f1"), ("a", "f2"),
            ("b", "c"), ("b", "f5"), ("c", "f3"), ("c", "f4")]
for pai, filho in ligacoes:
    x_pai, y_pai, _ = nos[pai]
    x_filho, y_filho, _ = nos[filho]
    painel_dir.plot([x_pai, x_filho], [y_pai, y_filho], color="black", zorder=1)
for nome, (x, y, rotulo) in nos.items():
    # nó interno com moldura; folha sem moldura
    borda = "none" if nome.startswith("f") else "black"
    painel_dir.text(x, y, rotulo, ha="center", va="center", fontsize=12,
                    bbox={"facecolor": "white", "edgecolor": borda})
painel_dir.set_xlim(0, 1)
painel_dir.set_ylim(-0.35, 1.1)
painel_dir.axis("off")

plt.tight_layout()
plt.show()

(a) Esboce a árvore que corresponde à partição do espaço dos preditores do painel da esquerda. Escreva a árvore na célula de texto como uma lista aninhada: cada nó interno é uma condição, com o ramo do *sim* e o do *não* indentados logo abaixo dela, e cada folha é a média de $Y$ na região. Uma árvore com um corte só, em $X_1 < 4$, com média 1 onde a condição vale e 7 onde não vale, ficaria assim:

- $X_1 < 4$
    - sim: 1
    - não: 7

*sua resposta aqui*

(b) Crie um diagrama parecido com o painel da esquerda, usando a árvore do painel da direita. Divida o espaço dos preditores nas regiões corretas e indique a média de cada região. A célula que desenhou as duas figuras serve de modelo.

In [ ]:
# sua resposta

## Exercício 4

*Do exercício 5 do capítulo 8 de James et al. (2023).*

Suponha que produzimos dez amostras bootstrap de um conjunto de dados com duas classes, vermelha e verde. Ajustamos uma árvore de classificação a cada amostra bootstrap e, para um valor específico de $X$, obtemos 10 estimativas de $\Pr(\text{a classe é vermelha} \mid X)$:

$$
0{,}1;\quad 0{,}15;\quad 0{,}2;\quad 0{,}2;\quad 0{,}55;\quad 0{,}6;\quad 0{,}6;\quad 0{,}65;\quad 0{,}7;\quad 0{,}75.
$$

Há duas formas comuns de combinar esses resultados numa única previsão de classe. Uma é o voto majoritário, da [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-bagging.html). A outra é classificar pela probabilidade média, como na [seção 13.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-erro-out-of-bag.html). Neste exemplo, qual é a classificação final em cada uma das duas formas?

*sua resposta aqui*

## Exercício 5

*Do exercício 8 do capítulo 8 de James et al. (2023).*

Neste exercício, você vai prever `vendas` no conjunto `Carseats`, em `dados/Carseats.csv`, com árvores de regressão e os métodos ligados a elas, tratando a resposta como variável quantitativa. O conjunto é simulado e descreve as vendas de cadeirinhas infantis para carro em 400 lojas. As variáveis são:

- `vendas`: unidades vendidas na loja, em milhares
- `preco_concorrente`: preço cobrado pelo concorrente na mesma praça
- `renda`: nível de renda da comunidade, em milhares de dólares
- `propaganda`: verba local de propaganda da empresa na loja, em milhares de dólares
- `populacao`: população da região, em milhares
- `preco`: preço que a empresa cobra pela cadeirinha na loja
- `prateleira`: qualidade da posição da cadeirinha na prateleira da loja — `ruim`, `média` ou `boa`
- `idade_media`: idade média da população local
- `escolaridade`: nível de escolaridade da região
- `urbana`: se a loja fica em área urbana (`sim` ou `não`)
- `eua`: se a loja fica nos Estados Unidos (`sim` ou `não`)

As colunas `prateleira`, `urbana` e `eua` são categóricas. Troque-as por indicadoras de todas as categorias, com `pd.get_dummies(..., dtype=float)`, como na [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-arvores-de-classificacao.html), deixando `vendas` de fora de $X$.

Use como erro de teste o MSE no conjunto de teste, com `mean_squared_error`. Use `random_state=3` na divisão, em toda árvore, em todo `KFold` e em toda floresta.

(a) Divida os dados em um conjunto de treino e um de teste, com `train_test_split(..., test_size=0.5, random_state=3)`. No texto, diga quantas lojas ficaram em cada conjunto.

In [ ]:
# sua resposta

*sua resposta aqui*

(b) Ajuste uma árvore de regressão aos dados de treino. Faça um gráfico da árvore e interprete os resultados. Que MSE de teste você obtém? Ajuste a árvore sem limite de tamanho, `DecisionTreeRegressor(random_state=3)`, como a árvore grande da [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-podando-a-arvore.html). Ela tem folhas demais para um gráfico legível: desenhe só os primeiros níveis, com `plot_tree(..., max_depth=2)`, e diga no texto quantas folhas a árvore inteira tem, com `get_n_leaves()`.

In [ ]:
# sua resposta

*sua resposta aqui*

(c) Use validação cruzada para escolher o nível de complexidade da árvore. A poda melhora o MSE de teste? Faça como na [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-podando-a-arvore.html): tire os valores de `ccp_alpha` do `cost_complexity_pruning_path` da árvore do item (b), monte a grade com um ponto dentro de cada intervalo e escolha `ccp_alpha` com `GridSearchCV` e `KFold(10, shuffle=True, random_state=3)`. No texto, diga quantas folhas tem a árvore escolhida.

In [ ]:
# sua resposta

*sua resposta aqui*

(d) Use o bagging para analisar estes dados. Que MSE de teste você obtém? Use os valores de `feature_importances_` para dizer quais variáveis são as mais importantes. O bagging é a floresta com $m = p$, `RandomForestRegressor(n_estimators=500, max_features=None, random_state=3)`, como na [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-florestas-aleatorias.html); a importância por redução de impureza é a da [seção 13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-importancia-de-variaveis.html).

In [ ]:
# sua resposta

*sua resposta aqui*

(e) Use florestas aleatórias para analisar estes dados. Que MSE de teste você obtém? Use os valores de `feature_importances_` para dizer quais variáveis são as mais importantes. Descreva o efeito de $m$, o número de variáveis consideradas em cada divisão, sobre o MSE de teste obtido. Para o MSE de teste e as importâncias, use a escolha usual da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-florestas-aleatorias.html), $m \approx \sqrt{p}$, com `RandomForestRegressor(n_estimators=500, max_features="sqrt", random_state=3)`. Para o efeito de $m$, ajuste `RandomForestRegressor(n_estimators=500, max_features=m, random_state=3)` para cada $m$ de 1 a $p$, num laço, e mostre o MSE de teste de cada $m$ numa tabela ou num gráfico.

In [ ]:
# sua resposta

*sua resposta aqui*

## Exercício 6

*Do exercício 9 do capítulo 8 de James et al. (2023).*

Este exercício usa o conjunto `OJ`, em `dados/OJ.csv`, com 1.070 compras de suco de laranja em que o cliente levou a marca Citrus Hill (`CH`) ou a Minute Maid (`MM`). As variáveis são:

- `compra`: a marca comprada, `CH` ou `MM`
- `semana`: a semana da compra
- `id_loja`: o código da loja — 1, 2, 3, 4 ou 7
- `preco_ch`, `preco_mm`: o preço de tabela de cada marca
- `desconto_ch`, `desconto_mm`: o desconto oferecido em cada marca
- `promocao_ch`, `promocao_mm`: 1 se a marca estava em promoção especial, 0 se não
- `fidelidade_ch`: a fidelidade do cliente à marca Citrus Hill, entre 0 e 1
- `preco_final_ch`, `preco_final_mm`: o preço de cada marca, já com o desconto
- `diferenca_preco`: o preço final da MM menos o da CH
- `loja_7`: se a compra foi na loja 7 (`sim` ou `não`)
- `perc_desconto_ch`, `perc_desconto_mm`: o desconto dividido pelo preço de tabela
- `diferenca_preco_tabela`: o preço de tabela da MM menos o da CH
- `loja`: o código da loja, de 0 a 4, em que a loja 7 vira 0

A coluna `loja_7` é categórica. Troque-a por indicadoras, com `pd.get_dummies(..., dtype=float)`, deixando `compra` de fora de $X$. Use `random_state=3` na divisão, em toda árvore e em todo `StratifiedKFold`.

(a) Crie um conjunto de treino com uma amostra aleatória de 800 observações, e um conjunto de teste com as observações restantes. Use `train_test_split(..., train_size=800, random_state=3)`.

In [ ]:
# sua resposta

(b) Ajuste uma árvore aos dados de treino, com `compra` como resposta e as outras variáveis como preditores. Ajuste a árvore sem limite de tamanho, `DecisionTreeClassifier(random_state=3)`. Qual é a taxa de erro de treino?

In [ ]:
# sua resposta

*sua resposta aqui*

(c) Faça um gráfico da árvore e interprete os resultados. Como no exercício 5, desenhe só os primeiros níveis, com `plot_tree(..., max_depth=2)`. Quantas folhas a árvore tem?

In [ ]:
# sua resposta

*sua resposta aqui*

(d) Use a função `export_text` para produzir um resumo da árvore ajustada em texto. Escolha uma das folhas e interprete a informação mostrada. A árvore inteira ocupa centenas de linhas: use `export_text(arvore, feature_names=list(X.columns), show_weights=True, max_depth=3)`, que mostra só os primeiros níveis, e escolha uma das folhas que aparecem nessa saída, as linhas com `weights` e `class`. A ordem dos dois números em `weights` é a de `arvore.classes_`.

In [ ]:
# sua resposta

*sua resposta aqui*

(e) Preveja a resposta nos dados de teste e produza uma matriz de confusão que compare os rótulos de teste com os rótulos previstos, com `confusion_matrix(..., labels=["CH", "MM"])`, como na [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-arvores-de-classificacao.html). Qual é a taxa de erro de teste?

In [ ]:
# sua resposta

*sua resposta aqui*

(f) Use validação cruzada no conjunto de treino para escolher o tamanho da árvore. Faça como na [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-arvores-de-classificacao.html): grade de `ccp_alpha` tirada do `cost_complexity_pruning_path` da árvore do item (b), em ordem decrescente, e `GridSearchCV` com `StratifiedKFold(10, shuffle=True, random_state=3)` e `scoring="accuracy"`.

In [ ]:
# sua resposta

(g) Faça um gráfico com o tamanho da árvore no eixo horizontal e a taxa de erro de classificação de validação cruzada no eixo vertical. O tamanho é o número de folhas da árvore ajustada no treino com cada `ccp_alpha` da grade; a taxa de erro é 1 menos a acurácia média, `cv_results_["mean_test_score"]`.

In [ ]:
# sua resposta

(h) Que tamanho de árvore corresponde à menor taxa de erro de validação cruzada?

In [ ]:
# sua resposta

*sua resposta aqui*

(i) Produza uma árvore podada com o tamanho escolhido pela validação cruzada. Se a validação cruzada não escolher uma árvore podada, crie uma árvore podada com cinco folhas.

In [ ]:
# sua resposta

(j) Compare as taxas de erro de treino da árvore podada e da não podada. Qual é maior?

In [ ]:
# sua resposta

*sua resposta aqui*

(k) Compare as taxas de erro de teste da árvore podada e da não podada. Qual é maior?

In [ ]:
# sua resposta

*sua resposta aqui*